# Bridge / chorus: audit split train

Buka dari root **SongMap**. Notebook memakai Python `.venv-songformer311` di sibling Section-Annotations secara eksplisit; kernel notebook boleh berbeda.

Memakai checkpoint PyTorch `worship-v3/best.pt`, cache fitur encoder asli, dan section train yang sudah disetujui. Ini bukan benchmark model Core ML palette6 di aplikasi. Tidak menjalankan encoder, membaca/decoding audio, mengubah label, atau memulai training. Validation/test tidak menjadi target audit.

Run pertama satu lagu, lalu seluruh train (sebelumnya 251 lagu). Per-lagu disimpan agar bisa dilanjutkan. Audit train mengukur pola fitting, bukan generalisasi.

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess, json
assert Path('scripts/audit_train_bridge.py').is_file(), 'Buka notebook dari root SongMap'
SONGSTRUCT_ROOT=Path('../Section-Annotations').resolve()
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'
CHECKPOINT=SONGSTRUCT_ROOT/'outputs.local/finetuning/runs/worship-v3/best.pt'
assert RUN_PYTHON.is_file(), f'Python tidak ditemukan: {RUN_PYTHON}'
assert CHECKPOINT.is_file(), f'Checkpoint tidak ditemukan: {CHECKPOINT}'
DEVICE='mps'
output=Path('outputs.local')/('train-bridge-audit-'+datetime.now().strftime('%Y%m%d-%H%M%S'))
subprocess.run([str(RUN_PYTHON),'-c',"import sys,importlib.util; missing=[p for p in ('numpy','torch','scipy','omegaconf','safetensors','ema_pytorch') if importlib.util.find_spec(p) is None]; print(sys.executable); sys.exit('Dependency missing: '+', '.join(missing)) if missing else print('Dependency check OK')"],check=True)
print('Checkpoint:',CHECKPOINT)
print('Output:',output)


In [ ]:
def run_audit(limit,name):
    output.mkdir(parents=True,exist_ok=True)
    log=output/(name+'.log')
    command=[str(RUN_PYTHON),'-u','scripts/audit_train_bridge.py',
        '--songstruct-root',str(SONGSTRUCT_ROOT),'--checkpoint',str(CHECKPOINT),
        '--device',DEVICE,'--limit',str(limit),'--output-dir',str(output)]
    with log.open('w',encoding='utf-8') as handle:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:
        raise RuntimeError(f'Gagal ({status}); lihat {log.resolve()}. Cache lagu yang selesai tetap tersedia.')
run_audit(1,'one')


In [ ]:
run_audit(0,'all')
r=json.loads((output/'train-all.json').read_text())
for key in ('split','full_train','tracks','eligible_sections','manual_label_counts',
            'manual_bridge_count','manual_bridge_predicted_as','bridge_predictions_by_relative_position',
            'bridge_to_chorus_count','chorus_to_bridge_count','affected_tracks',
            'high_confidence_wrong_cases_ge_0_8','excluded_sections_not_targets','limitations'):
    print(key+':',json.dumps(r[key],indent=2))
print('Contoh prioritas (diagnostik, bukan perintah relabel):')
for c in r['cases'][:10]:
    print(c['track_id'],round(c['start'],2),round(c['end'],2),c['label_manual'],'->',c['prediction'],round(c['confidence'],3))
print('Bagikan:',(output/'train-all.json').resolve())


Label manual tetap menjadi referensi. Confidence tinggi pada prediksi yang berbeda bukan bukti anotasi salah. Tetangga yang dilaporkan adalah section layak terdekat; gap disimpan agar section yang ditahan tidak diasumsikan bersebelahan.

Jika train bridge sudah baik tetapi validation buruk, menambah epoch saja berisiko memperbesar overfitting. Jika kesalahan juga dominan pada train, audit akan membantu membedakan keterbatasan fitting, konteks dan distribusi target sebelum memilih eksperimen berikutnya. Tidak ada keputusan model atau perbaikan label otomatis.